The notebooks establishes data preprocessing procedures and builds an initial model that will act as baseline for future comparisons.

In [1]:
import pandas as pd
import pickle as pkl
import importlib
import lightgbm

from helpers import table_navigator
from helpers import preprocessing
from helpers.utils import sklearn_helper
from helpers import submissions

# Preprocessing

## Applications (main) table

### Manual applications preprocessor loading
**note** - these steps showcase how the background preprocessor steps works to provide context in later notebooks and can be easily skipped.

We first load the data

In [2]:
app_train = table_navigator.get_tables_from_dir('data/raw_csv', 'application_train')['application_train']
app_train.tail(3)

Loading tables: ['application_train.csv']
loaded application_train with shape (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,0,0,0,0,1.0,0.0,0.0,1.0,0.0,1.0
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
307510,456255,0,Cash loans,F,N,N,0,157500.0,675000.0,49117.5,...,0,0,0,0,0.0,0.0,0.0,2.0,0.0,1.0


We then load preanalysed columns from nb1. 

In [3]:
with open('data/processed/application_columns.pkl', 'rb') as file:
    application_columns = pkl.load(file)


application_columns
cat_columns = preprocessing.column_list_adapter(application_columns['cat_columns'])
num_columns = preprocessing.column_list_adapter(application_columns['num_columns'])
bool_columns = preprocessing.column_list_adapter(application_columns['bool_columns'])
str_columns = preprocessing.column_list_adapter(application_columns['str_columns'])


In [4]:
print(cat_columns)

['WALLSMATERIAL_MODE', 'HOUSETYPE_MODE', 'FONDKAPREMONT_MODE', 'NAME_EDUCATION_TYPE', 'NAME_CONTRACT_TYPE', 'NAME_HOUSING_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'NAME_TYPE_SUITE', 'ORGANIZATION_TYPE', 'EMERGENCYSTATE_MODE', 'CODE_GENDER', 'OCCUPATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_INCOME_TYPE']


In [5]:
print(num_columns)

['FLOORSMIN_MODE', 'YEARS_BUILD_AVG', 'HOUR_APPR_PROCESS_START', 'APARTMENTS_MODE', 'ELEVATORS_AVG', 'LIVINGAREA_AVG', 'NONLIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'REGION_RATING_CLIENT', 'COMMONAREA_AVG', 'AMT_REQ_CREDIT_BUREAU_DAY', 'YEARS_BEGINEXPLUATATION_MODE', 'LIVINGAREA_MEDI', 'AMT_REQ_CREDIT_BUREAU_HOUR', 'CNT_CHILDREN', 'YEARS_BUILD_MEDI', 'LANDAREA_AVG', 'COMMONAREA_MODE', 'OBS_60_CNT_SOCIAL_CIRCLE', 'AMT_GOODS_PRICE', 'REGION_POPULATION_RELATIVE', 'FLOORSMAX_MODE', 'AMT_ANNUITY', 'ENTRANCES_AVG', 'FLOORSMAX_MEDI', 'YEARS_BEGINEXPLUATATION_AVG', 'EXT_SOURCE_2', 'LANDAREA_MEDI', 'BASEMENTAREA_MODE', 'DAYS_ID_PUBLISH', 'COMMONAREA_MEDI', 'DAYS_BIRTH', 'LIVINGAPARTMENTS_AVG', 'AMT_INCOME_TOTAL', 'DAYS_REGISTRATION', 'CNT_FAM_MEMBERS', 'TOTALAREA_MODE', 'EXT_SOURCE_3', 'FLOORSMAX_AVG', 'BASEMENTAREA_AVG', 'FLOORSMIN_MEDI', 'DAYS_EMPLOYED', 'NONLIVINGAREA_MODE', 'FLOORSMIN_AVG', 'OBS_30_CNT_SOCIAL_CIRCLE', 'LANDAREA_MODE', 'AMT_REQ_CREDIT_BUREAU_YEAR', 'BASEMENTAREA_MEDI', 'RE

In [6]:
print(bool_columns)

['FLAG_DOCUMENT_9', 'FLAG_WORK_PHONE', 'FLAG_DOCUMENT_12', 'FLAG_MOBIL', 'TARGET', 'FLAG_DOCUMENT_4', 'FLAG_DOCUMENT_16', 'FLAG_PHONE', 'REG_CITY_NOT_WORK_CITY', 'FLAG_DOCUMENT_6', 'FLAG_DOCUMENT_21', 'FLAG_DOCUMENT_8', 'REG_CITY_NOT_LIVE_CITY', 'FLAG_DOCUMENT_20', 'FLAG_EMP_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_DOCUMENT_13', 'REG_REGION_NOT_WORK_REGION', 'FLAG_DOCUMENT_14', 'FLAG_DOCUMENT_10', 'FLAG_DOCUMENT_2', 'FLAG_EMAIL', 'FLAG_OWN_REALTY', 'LIVE_CITY_NOT_WORK_CITY', 'FLAG_DOCUMENT_7', 'FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_11', 'FLAG_OWN_CAR', 'FLAG_DOCUMENT_19', 'REG_REGION_NOT_LIVE_REGION', 'FLAG_DOCUMENT_17', 'FLAG_DOCUMENT_15', 'FLAG_DOCUMENT_5', 'LIVE_REGION_NOT_WORK_REGION', 'FLAG_DOCUMENT_18']


We then initiate the preprocessor, and check whether we have covered all columns.

In [7]:
processor = preprocessing.Preprocessor(
    data=app_train,
    cat_columns=cat_columns,
    num_columns=num_columns,
    bool_columns=bool_columns,
    str_columns=str_columns
)

processor.check_columns();

No missing columns.
Unexpected columns: ['SK_ID_CURR']


### Automated preprocessor loading loading
To save time in future notebooks, applications table can be loaded by a single function call function:

In [8]:
processor = preprocessing.load_pkl_to_preprocessor('application_train')
processor.check_columns();

Loading tables: ['application_train.csv']
loaded application_train with shape (307511, 122)
No missing columns.
Unexpected columns: ['SK_ID_CURR']


All the informative columns were captured successfully. The `SK_ID_CURR` is non an informative column and was thus excluded.

### Data type conversion

In [9]:
processor.convert_all('category').info()

No missing columns.
Unexpected columns: ['SK_ID_CURR']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 122 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: bool(35), category(14), float64(65), int64(8)
memory usage: 185.6 MB


The dataset maintained appropriate columns after conversion

In [10]:
processor.check_conversion_success()

All columns have the expected data types.


And the manual column grouping from nb1 and nb2 is:

In [11]:
print(processor.column_structure)

{'core_identification': ['SK_ID_CURR', 'TARGET'], 'loan_basics': ['NAME_CONTRACT_TYPE', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE'], 'financial_info': ['AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE'], 'assets': ['FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'OWN_CAR_AGE'], 'time_features': ['DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE', 'HOUR_APPR_PROCESS_START'], 'contact_flags': ['FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL'], 'demographics': ['CODE_GENDER', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'DAYS_BIRTH'], 'external_sources': ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'], 'location_features': ['REGION_POPULATION_RELATIVE', 'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION', 'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY'], 'social_circle': ['OBS_30_CNT_SOCI

The dataset is now ready to be used for LightGBM model training.

## Preprocessing Supplementary tables
Identical to the main applications table, the supplementary tables can be loaded and prepared with just a few commands.

#### Bureau

In [12]:
importlib.reload(preprocessing)
bureau = preprocessing.load_pkl_to_preprocessor('bureau')
bureau.convert_all('category').info()
print(bureau.column_structure)
del bureau

Loading tables: ['bureau.csv']
loaded bureau with shape (1716428, 17)
dict_keys(['credit_status', 'bureau_overdue', 'time_variables', 'bureau', 'bureau_balance', 'pos_cash_balance', 'balance_limits', 'drawings_payments', 'contract_installments', 'credit_card_balance', 'previous_application', 'installments_payments'])
No missing columns.
Unexpected columns: ['SK_ID_CURR', 'SK_ID_BUREAU']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1716428 entries, 0 to 1716427
Data columns (total 17 columns):
 #   Column                  Dtype   
---  ------                  -----   
 0   SK_ID_CURR              int64   
 1   SK_ID_BUREAU            int64   
 2   CREDIT_ACTIVE           category
 3   CREDIT_CURRENCY         category
 4   DAYS_CREDIT             int64   
 5   CREDIT_DAY_OVERDUE      int64   
 6   DAYS_CREDIT_ENDDATE     float64 
 7   DAYS_ENDDATE_FACT       float64 
 8   AMT_CREDIT_MAX_OVERDUE  float64 
 9   CNT_CREDIT_PROLONG      int64   
 10  AMT_CREDIT_SUM          float64 
 11

#### bureau_balance

In [13]:
credit_balance = preprocessing.load_pkl_to_preprocessor('bureau_balance')
credit_balance.convert_all('category').info()
print(credit_balance.column_structure)
del credit_balance

Loading tables: ['bureau_balance.csv']
loaded bureau_balance with shape (27299925, 3)
dict_keys(['credit_status', 'bureau_overdue', 'time_variables', 'bureau', 'bureau_balance', 'pos_cash_balance', 'balance_limits', 'drawings_payments', 'contract_installments', 'credit_card_balance', 'previous_application', 'installments_payments'])
No missing columns.
Unexpected columns: ['SK_ID_BUREAU']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27299925 entries, 0 to 27299924
Data columns (total 3 columns):
 #   Column          Dtype   
---  ------          -----   
 0   SK_ID_BUREAU    int64   
 1   MONTHS_BALANCE  int64   
 2   STATUS          category
dtypes: category(1), int64(2)
memory usage: 442.6 MB
['SK_ID_BUREAU', 'MONTHS_BALANCE', 'STATUS']


#### credit_card_balance

In [14]:
credit_card_balance = preprocessing.load_pkl_to_preprocessor('credit_card_balance')
credit_card_balance.convert_all('category').info()
print(credit_card_balance.column_structure)
del credit_card_balance

Loading tables: ['credit_card_balance.csv']
loaded credit_card_balance with shape (3840312, 23)
dict_keys(['credit_status', 'bureau_overdue', 'time_variables', 'bureau', 'bureau_balance', 'pos_cash_balance', 'balance_limits', 'drawings_payments', 'contract_installments', 'credit_card_balance', 'previous_application', 'installments_payments'])
No missing columns.
Unexpected columns: ['SK_ID_PREV', 'SK_ID_CURR']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3840312 entries, 0 to 3840311
Data columns (total 23 columns):
 #   Column                      Dtype   
---  ------                      -----   
 0   SK_ID_PREV                  int64   
 1   SK_ID_CURR                  int64   
 2   MONTHS_BALANCE              int64   
 3   AMT_BALANCE                 float64 
 4   AMT_CREDIT_LIMIT_ACTUAL     int64   
 5   AMT_DRAWINGS_ATM_CURRENT    float64 
 6   AMT_DRAWINGS_CURRENT        float64 
 7   AMT_DRAWINGS_OTHER_CURRENT  float64 
 8   AMT_DRAWINGS_POS_CURRENT    float64 
 9   AMT_IN

#### POS_CASH_balance

In [15]:
import importlib
importlib.reload(preprocessing)
pos_cash_balance = preprocessing.load_pkl_to_preprocessor('POS_CASH_balance')
pos_cash_balance.convert_all('category').info()
print(pos_cash_balance.column_structure)
del pos_cash_balance

Loading tables: ['POS_CASH_balance.csv']
loaded POS_CASH_balance with shape (10001358, 8)
dict_keys(['credit_status', 'bureau_overdue', 'time_variables', 'bureau', 'bureau_balance', 'pos_cash_balance', 'balance_limits', 'drawings_payments', 'contract_installments', 'credit_card_balance', 'previous_application', 'installments_payments'])
No missing columns.
Unexpected columns: ['SK_ID_PREV', 'SK_ID_CURR']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10001358 entries, 0 to 10001357
Data columns (total 8 columns):
 #   Column                 Dtype   
---  ------                 -----   
 0   SK_ID_PREV             int64   
 1   SK_ID_CURR             int64   
 2   MONTHS_BALANCE         int64   
 3   CNT_INSTALMENT         float64 
 4   CNT_INSTALMENT_FUTURE  float64 
 5   NAME_CONTRACT_STATUS   category
 6   SK_DPD                 int64   
 7   SK_DPD_DEF             int64   
dtypes: category(1), float64(2), int64(5)
memory usage: 543.7 MB
['SK_ID_PREV', 'SK_ID_CURR', 'MONTHS_BALANC

#### previous_application

In [16]:
previous_application = preprocessing.load_pkl_to_preprocessor('previous_application')
previous_application.convert_all('category').info()
print(previous_application.column_structure)
del previous_application

Loading tables: ['previous_application.csv']
loaded previous_application with shape (1670214, 37)
dict_keys(['credit_status', 'bureau_overdue', 'time_variables', 'bureau', 'bureau_balance', 'pos_cash_balance', 'balance_limits', 'drawings_payments', 'contract_installments', 'credit_card_balance', 'previous_application', 'installments_payments'])
No missing columns.
Unexpected columns: ['SK_ID_PREV', 'SK_ID_CURR']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1670214 entries, 0 to 1670213
Data columns (total 37 columns):
 #   Column                       Non-Null Count    Dtype   
---  ------                       --------------    -----   
 0   SK_ID_PREV                   1670214 non-null  int64   
 1   SK_ID_CURR                   1670214 non-null  int64   
 2   NAME_CONTRACT_TYPE           1670214 non-null  category
 3   AMT_ANNUITY                  1297979 non-null  float64 
 4   AMT_APPLICATION              1670214 non-null  float64 
 5   AMT_CREDIT                   1670213 no

#### installments_payments

In [17]:
installments_payments = preprocessing.load_pkl_to_preprocessor('installments_payments')
installments_payments.convert_all('category').info()
print(installments_payments.column_structure)
del installments_payments

Loading tables: ['installments_payments.csv']
loaded installments_payments with shape (13605401, 8)
dict_keys(['credit_status', 'bureau_overdue', 'time_variables', 'bureau', 'bureau_balance', 'pos_cash_balance', 'balance_limits', 'drawings_payments', 'contract_installments', 'credit_card_balance', 'previous_application', 'installments_payments'])
No missing columns.
Unexpected columns: ['SK_ID_PREV', 'SK_ID_CURR']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13605401 entries, 0 to 13605400
Data columns (total 8 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_PREV              int64  
 1   SK_ID_CURR              int64  
 2   NUM_INSTALMENT_VERSION  float64
 3   NUM_INSTALMENT_NUMBER   int64  
 4   DAYS_INSTALMENT         float64
 5   DAYS_ENTRY_PAYMENT      float64
 6   AMT_INSTALMENT          float64
 7   AMT_PAYMENT             float64
dtypes: float64(5), int64(3)
memory usage: 830.4 MB
['SK_ID_PREV', 'SK_ID_CURR', 'NUM_INSTALMENT_V

# Baseline model training

ML model performance will be focus of future notebooks and EDA. Consequently, a non-engineered model baseline will be established for future comparison.

In [18]:
X_train = processor.get_expected_df()
y_train = X_train.pop('TARGET')


X_train.info()

print()
y_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 120 entries, WALLSMATERIAL_MODE to FLAG_DOCUMENT_18
dtypes: bool(34), category(14), float64(65), int64(7)
memory usage: 183.0 MB

<class 'pandas.core.series.Series'>
RangeIndex: 307511 entries, 0 to 307510
Series name: TARGET
Non-Null Count   Dtype
--------------   -----
307511 non-null  bool 
dtypes: bool(1)
memory usage: 300.4 KB


#### Test data

In [19]:
targetless_list = bool_columns.copy()
targetless_list.remove('TARGET')

test_processor = preprocessing.load_pkl_to_preprocessor('application_test')
test_processor.check_columns()
test_processor.convert_all('category')

X_test_id = test_processor.data.pop('SK_ID_CURR')
X_test = test_processor.get_expected_df()

print()
X_test.info()

Loading tables: ['application_test.csv']
loaded application_test with shape (48744, 121)
No missing columns.
Unexpected columns: ['SK_ID_CURR']
No missing columns.
Unexpected columns: ['SK_ID_CURR']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48744 entries, 0 to 48743
Columns: 120 entries, WALLSMATERIAL_MODE to FLAG_DOCUMENT_18
dtypes: bool(34), category(14), float64(65), int64(7)
memory usage: 29.0 MB


#### Metric choice

In [20]:
y_train.value_counts(normalize=True)

TARGET
False    0.919271
True     0.080729
Name: proportion, dtype: float64

Due to the dataset being highly imbalanced (only 9% minority group), a more fine metric is needed. Consequently, the following metric will be used:
* Receiver Operator Curve AUC - original metric used during the contest.
* Precision-Recall curve AUC - more sensitive towards imbalanced data.
* f1_macro - simplest to interpret

#### Model Choice
The data:
* Includes numeric, boolean, and categorical variables.
* Includes large proportion of missing values for some features.
* Is highly asymmetric for response variable.
* Highly tabular
* Following data cleaning in NB1 and NB2, problematic for regression style models.

Consequently, **gradient boosted tree model from lightGBM library will be implemented** 

In [21]:

importlib.reload(sklearn_helper)

sklearn_helper.stratified_cv_model(
    lightgbm.LGBMClassifier(is_unbalance=True, random_state=3, verbose=-1), 
    X_train, 
    y_train, 
    scoring=['average_precision', 'roc_auc', 'f1_macro']
)

,average_precision,roc_auc,f1_macro
0,0.2408,0.7568,0.5428


We then confirm that the model is not significantly overfit by submitting it for the context.

In [22]:
submissions.prepare_submission(
    lightgbm.LGBMClassifier(is_unbalance=True, random_state=3, verbose=-1).fit(X_train, y_train), 
    X_test, 
    X_test_id,
    'lgbm_basic'
)

Submission created: submissions\sub_lgbm_basic_2025-07-12_10-26.csv


'submissions\\sub_lgbm_basic_2025-07-12_10-26.csv'

The contest score is 0.74262, which is close enough to the training ROC_AUC to not cause concerns.